In [ ]:
%matplotlib widget
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC, SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample
import matplotlib.style
#matplotlib.style.use("nf_style.mplstyle")
import matplotlib.pyplot as plt
import numpy as np

import matplotlib as mpl
from svgpathtools import svg2paths
from svgpath2mpl import parse_path



def generate_marker_from_svg(svg_path):
    image_path, attributes = svg2paths(svg_path)

    image_marker = parse_path(attributes[0]['d'])

    image_marker.vertices -= image_marker.vertices.mean(axis=0)

    image_marker = image_marker.transformed(mpl.transforms.Affine2D().rotate_deg(180))
    image_marker = image_marker.transformed(mpl.transforms.Affine2D().scale(-1,1))

    return image_marker



markers = [
    # '.',
    generate_marker_from_svg('heart-suit.svg'),
    generate_marker_from_svg('hourglass.svg'),
    # ',',
    'o',
    'v',
    '^',
    '<',
    '>',
    '8',
    's',
    'p',
    '*',
    'h',
    'H',
    'D',
    'd',
    'P',
    'X',
    # '$\\alpha$',
    # '$\\beta$',
    # '$\gamma$',
    # '$\sigma$',
    # '$\infty$',
    '$\spadesuit$',
    # '$\heartsuit$',
    # '$\diamondsuit$',
    '$\clubsuit$',
    # '$\\bigodot$', 
    # '$\\bigotimes$', 
    # '$\\bigoplus$', 
    # '$\imath$', 
    # '$\\bowtie$',
    # '$\\bigtriangleup$', 
    # '$\\bigtriangledown$', 
    # '$\oslash$',
    # '\ast', 
    # '$\\times$', 
    # '$\circ$', 
    # '$\\bullet$', 
    # '$\star$', 
    '+',
    '$\Theta$', 
    '$\Xi$', 
    '$\Phi$',
    '$\$$', 
    '$\#$', 
    '$\%$', 
    '$\S$',
    ]



def print_all_unique_elements_in_column(file_path, column_name):
    try:
        # Load the CSV file into a pandas DataFrame
        df = pd.read_csv(file_path)

        # Check if the specified column exists in the DataFrame
        if column_name not in df.columns:
            print(f"Error: Column '{column_name}' not found in the CSV file.")
            return

        # Extract and print all unique elements in the specified column
        unique_elements = df[column_name].unique()
        print(f"Unique elements in '{column_name}':")
        for element in unique_elements:
            print(element)

        return unique_elements

    except FileNotFoundError:
        print(f"Error: File '{file_path}' not found.")
    except pd.errors.EmptyDataError:
        print(f"Error: File '{file_path}' is empty.")
    except pd.errors.ParserError:
        print(f"Error: Unable to parse the CSV file '{file_path}'. Check if it's a valid CSV file.")


# Analytical least squares solution for y = ax + bw + c
def analytical_least_squares(x, w, y, sigma=None):
    
    # If sigma is not provided, assume that all points have the same weight
    if sigma is None:
        sigma = np.ones_like(x)

    sigma = np.array(sigma) ** 2

    # Definitions and auxiliary variables
    X = np.sum(x/sigma)
    W = np.sum(w/sigma)
    X2 = np.sum(x**2/sigma)
    W2 = np.sum(w**2/sigma)
    XW = np.sum(x*w/sigma)
    Y = np.sum(y/sigma)
    XY = np.sum(x*y/sigma)
    WY = np.sum(w*y/sigma)
    n = np.sum(sigma**-1)

    # Auxiliar matrix A
    A = np.array([
        [X2, XW, X],
        [XW, W2, W],
        [ X,  W, n]
    ])

    # Auxiliar vector B
    B = np.array([XY, WY, Y])

    # Solve the system of equations
    a, b, c = np.matmul(np.linalg.inv(A), B)

    return a, b, c
    

In [ ]:
# file_path = 'database_finalissima.csv'
file_path = 'database_100ms.csv'
# file_path = 'database_refined.csv'

all_labels = print_all_unique_elements_in_column(file_path, 'label')

# Labels

In [ ]:
# Print all column names in the CSV file
df = pd.read_csv(file_path)
for name in df.columns:
    print(name) 

In [ ]:
print(f"KNOTS = {df['Nb of knots'].values[0]}")

In [ ]:
# Load the dataframe
df = pd.read_csv(file_path)

# Label column name
label_column_name = 'label'

# Class
class_A_labels = [label for label in all_labels if (("developed" in label.lower()) and ('eda' in label.lower()) and ('elmy' not in label.lower()))]
print(class_A_labels)

class_B_labels = [label for label in all_labels if (("elmy" in label.lower()) and ('early' not in label.lower()))]
print(class_B_labels)



# shots = [ # 0.8 MA
#     # 36257,
#     36157,
#     36124,
#     37193,
#     36349,
#     36151,
#     ]



# shots = [ # 0.7 MA
#     38049, 
#     38066, 
#     38067, # Shot todo comido?
#     38742, 
#     38743, 
#     38744,
#     ]

# Scan de corrente
shots = [
    36980, 
    35970, 
    38808, 
    38049, 
    36124, 
    36157
]

# Combine class A and class B labels
selected_labels = class_A_labels + class_B_labels

# Filter dataframe based on selected labels
df = df[df['shot'].isin(shots)]
df = df[df[label_column_name].isin(selected_labels)]

# Add new columns to the dataframe
# df['1-Shapiro'] = 1 - df['|shapiro(Ipolsola>-rolling_0.999_quantile_50ms(-Ipolsola))']
df['1-Shapiro'] = 1 - df['MAC|shapiro(Ipolsola)']

# Apply a threshold to the new column
df = df[df['1-Shapiro'] < 0.03]


# 3D Pedestal Fit

In [ ]:
# Select which columns in the dataframe will be the "features", i.e., the coordinates for the SVM classification (choose 2)

# first_coordinate = 'UVS|D_tot'
first_coordinate = "FPC|IpiFP"

possible_second_coordinates = [
    "|P_TOT-Pr_sep",
    "TOT|P_TOT",
]

possible_third_coordinates = [
    '|ne0',
    '|ne95',
    '|pe0',
    '|pe95',
    '|Te0',
    '|Te95',
]

# possible_third_coordinates = [
#     'TOT|H-1_corr',
#     'TOT|Wmhd',
#     'TOT|tau_tot',
#     '|p_avg*tau_tot',
# ]

results = {
    'first_coordinate': [],
    'second_coordinate': [],
    'third_coordinate': [],
    'gamma1': [],
    'gamma2': [],
    'k': [],
    'r2': [],
}

fig, axes = plt.subplots(len(possible_third_coordinates), 2, figsize=(6, 12))  

axes = axes.T.flatten()
ii = -1
x=[]
y=[]
for second_coordinate in possible_second_coordinates:
    for third_coordinate in possible_third_coordinates:

        ii += 1

        # Select features and labels
        X = df[[first_coordinate, second_coordinate, third_coordinate, "shot"]]

        X[first_coordinate] = np.log(X[first_coordinate])
        X[second_coordinate] = np.log(X[second_coordinate])
        X[third_coordinate] = np.log(X[third_coordinate])

        # Check for nans
        mask = \
            X[first_coordinate].isnull()   | \
            X[second_coordinate].isnull()  | \
            X[third_coordinate].isnull()   | \
            np.isinf(X[first_coordinate])  | \
            np.isinf(X[second_coordinate]) | \
            np.isinf(X[third_coordinate]) 
        X = X[~mask]



        # 3D plot
        # fig = plt.figure(figsize=(6, 6))

        # ax = fig.add_subplot(111, projection='3d')
        # ax.scatter(np.log(X[first_coordinate]), np.log(X[second_coordinate]), np.log(X[third_coordinate]), c='r', marker='o')
        # ax.set_xlabel("log" + first_coordinate)
        # ax.set_ylabel("log" + second_coordinate)
        # ax.set_zlabel("log" + third_coordinate)

        # plt.savefig(f"3D_{first_coordinate}_{second_coordinate}_{third_coordinate}_shots{shots}.pdf".replace('|', '_').replace('/', '_').replace('\\', '_').replace('%', '_').replace('*', '_').replace('>', '_'), dpi=300)




        sigma = np.ones(len(X[first_coordinate]))

        for i, shot in enumerate(shots):
            sigma[X['shot'] == shot] *= (len(sigma[X['shot'] == shot]))**0.5

        # Define a power-law fitting function by fitting the log of the data
        def power_law(X, gamma1, gamma2, k):
            x1, x2 = X
            return x1 * gamma1 + x2 * gamma2 + k

        # Fit a power-law to the data
        from scipy.optimize import curve_fit

        # try:
        popt, pcov = curve_fit(
            power_law, 
            (X[first_coordinate], X[second_coordinate]), 
            X[third_coordinate], 
            p0=[1, 1, 1],
            sigma=sigma,
            )
        # except RuntimeError:
        #     # Optionally, use the analytical least squares solution
        #     # Resample the data to have the same weight per shot
        #     popt = analytical_least_squares(X[first_coordinate], X[second_coordinate], X[third_coordinate])
        #     pcov = np.zeros((3, 3))

        # Given the optimal parameters, calculate Pearson's R-squared
        xx = np.exp(X[first_coordinate])**popt[0] * np.exp(X[second_coordinate])**popt[1]
        yy = np.exp(X[third_coordinate])
        r2 = np.corrcoef(xx, yy)[0, 1] ** 2

        # Plot the data and the power-law fit
        # plt.figure(figsize=(3, 3))
        plt.sca(axes[ii])
        xx = np.exp(X[first_coordinate])**popt[0] * np.exp(X[second_coordinate])**popt[1]
        yy = np.exp(X[third_coordinate])
        x.append(xx)
        y.append(yy)
        plt.plot(xx, yy, 'o', label='Data', alpha=0.1,)
        plt.plot(
            np.sort(xx), 
            np.sort(xx)*np.exp(popt[2]), 
            label=f'Fit: $y = {popt[1]:.2f}x^{popt[0]:.2f}$',
            )
        x.append(np.sort(xx),)
        y.append(np.sort(xx)*np.exp(popt[2]))
        plt.xlabel(f"{first_coordinate}^{popt[0]:.2f} * {second_coordinate}^{popt[1]:.2f}")
        plt.ylabel(third_coordinate)

        plt.grid(False, which='minor')

        plt.title(
            # f"{shots}\n" + 
            f"k = {np.exp(popt[2]):.2e}\n" 
            # f"Same weight per {'POINT' if (sigma == 1.).all() else 'SHOT'}\n" +
            #f"Covariance matrix for the exponents:\n{pcov[0:2, 0:2]}\n" + 
            #f"Coefficient of determination r^2: {r2:.2f}"
        )
        # plt.tight_layout()
        # plt.savefig(f"2D_{first_coordinate}_{second_coordinate}_{third_coordinate}_shots{shots}.pdf".replace('|', '_').replace('/', '_').replace('\\', '_').replace('%', '_').replace('*', '_').replace('>', '_'), dpi=300)

        # Save the results
        results['first_coordinate'].append(first_coordinate)
        results['second_coordinate'].append(second_coordinate)
        results['third_coordinate'].append(third_coordinate)
        results['gamma1'].append(popt[0])
        results['gamma2'].append(popt[1])
        results['k'].append(np.exp(popt[2]))
        results['r2'].append(r2)
        
# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(f"IDA_results_shots{shots}.csv", index=False)

plt.tight_layout()
plt.savefig(f"IDA_fit_grid_shots{shots}.pdf".replace('|', '_').replace('/', '_').replace('\\', '_').replace('%', '_').replace('*', '_').replace('>', '_').replace(' ', ''), dpi=300)

# 3D Sep Fit

In [ ]:
# Select which columns in the dataframe will be the "features", i.e., the coordinates for the SVM classification (choose 2)

# first_coordinate = 'UVS|D_tot'
first_coordinate = "FPC|IpiFP"

possible_second_coordinates = [
    "|P_TOT-Pr_sep",
    "TOT|P_TOT",
]
'''
possible_third_coordinates = [
    '|ne0',
    '|ne95',
    '|pe0',
    '|pe95',
    '|Te0',
    '|Te95',
]
'''

possible_third_coordinates = [
    '$T_{sep}$ (eV)',
    '$n_{sep}$ ($10^{19}$ m$^{-3}$)',
    '$p_{sep}$ (kPa)',
    '$T_{sep}^{(m)}$ (eV)',
    '$n_{sep}^{(m)}$ ($10^{19}$ m$^{-3}$)',
    '$p_{sep}^{(m)}$ (kPa)',
    '$p_{sep}$ (kPa)',    
]

# possible_third_coordinates = [
#     'TOT|H-1_corr',
#     'TOT|Wmhd',
#     'TOT|tau_tot',
#     '|p_avg*tau_tot',
# ]

results = {
    'first_coordinate': [],
    'second_coordinate': [],
    'third_coordinate': [],
    'gamma1': [],
    'gamma2': [],
    'k': [],
    'r2': [],
}

fig, axes = plt.subplots(len(possible_third_coordinates), 2, figsize=(6, 12))  

axes = axes.T.flatten()
ii = -1
x=[]
y=[]
for second_coordinate in possible_second_coordinates:
    for third_coordinate in possible_third_coordinates:

        ii += 1

        # Select features and labels
        X = df[[first_coordinate, second_coordinate, third_coordinate, "shot"]]

        X[first_coordinate] = np.log(X[first_coordinate])
        X[second_coordinate] = np.log(X[second_coordinate])
        X[third_coordinate] = np.log(X[third_coordinate])

        # Check for nans
        mask = \
            X[first_coordinate].isnull()   | \
            X[second_coordinate].isnull()  | \
            X[third_coordinate].isnull()   | \
            np.isinf(X[first_coordinate])  | \
            np.isinf(X[second_coordinate]) | \
            np.isinf(X[third_coordinate]) 
        X = X[~mask]



        # 3D plot
        # fig = plt.figure(figsize=(6, 6))

        # ax = fig.add_subplot(111, projection='3d')
        # ax.scatter(np.log(X[first_coordinate]), np.log(X[second_coordinate]), np.log(X[third_coordinate]), c='r', marker='o')
        # ax.set_xlabel("log" + first_coordinate)
        # ax.set_ylabel("log" + second_coordinate)
        # ax.set_zlabel("log" + third_coordinate)

        # plt.savefig(f"3D_{first_coordinate}_{second_coordinate}_{third_coordinate}_shots{shots}.pdf".replace('|', '_').replace('/', '_').replace('\\', '_').replace('%', '_').replace('*', '_').replace('>', '_'), dpi=300)




        sigma = np.ones(len(X[first_coordinate]))

        for i, shot in enumerate(shots):
            sigma[X['shot'] == shot] *= (len(sigma[X['shot'] == shot]))**0.5

        # Define a power-law fitting function by fitting the log of the data
        def power_law(X, gamma1, gamma2, k):
            x1, x2 = X
            return x1 * gamma1 + x2 * gamma2 + k

        # Fit a power-law to the data
        from scipy.optimize import curve_fit

        # try:
        popt, pcov = curve_fit(
            power_law, 
            (X[first_coordinate], X[second_coordinate]), 
            X[third_coordinate], 
            p0=[1, 1, 1],
            sigma=sigma,
            )
        # except RuntimeError:
        #     # Optionally, use the analytical least squares solution
        #     # Resample the data to have the same weight per shot
        #     popt = analytical_least_squares(X[first_coordinate], X[second_coordinate], X[third_coordinate])
        #     pcov = np.zeros((3, 3))

        # Given the optimal parameters, calculate Pearson's R-squared
        xx = np.exp(X[first_coordinate])**popt[0] * np.exp(X[second_coordinate])**popt[1]
        yy = np.exp(X[third_coordinate])
        r2 = np.corrcoef(xx, yy)[0, 1] ** 2

        # Plot the data and the power-law fit
        # plt.figure(figsize=(3, 3))
        plt.sca(axes[ii])
        xx = np.exp(X[first_coordinate])**popt[0] * np.exp(X[second_coordinate])**popt[1]
        yy = np.exp(X[third_coordinate])
        x.append(xx)
        y.append(yy)
        plt.plot(xx, yy, 'o', label='Data', alpha=0.1,)
        plt.plot(
            np.sort(xx), 
            np.sort(xx)*np.exp(popt[2]), 
            label=f'Fit: $y = {popt[1]:.2f}x^{popt[0]:.2f}$',
            )
        x.append(np.sort(xx),)
        y.append(np.sort(xx)*np.exp(popt[2]))
        plt.xlabel(f"{first_coordinate}^{popt[0]:.2f} * {second_coordinate}^{popt[1]:.2f}")
        plt.ylabel(third_coordinate)

        plt.grid(False, which='minor')

        plt.title(
            # f"{shots}\n" + 
            f"k = {np.exp(popt[2]):.2e}\n" 
            # f"Same weight per {'POINT' if (sigma == 1.).all() else 'SHOT'}\n" +
            #f"Covariance matrix for the exponents:\n{pcov[0:2, 0:2]}\n" + 
            #f"Coefficient of determination r^2: {r2:.2f}"
        )
        # plt.tight_layout()
        # plt.savefig(f"2D_{first_coordinate}_{second_coordinate}_{third_coordinate}_shots{shots}.pdf".replace('|', '_').replace('/', '_').replace('\\', '_').replace('%', '_').replace('*', '_').replace('>', '_'), dpi=300)

        # Save the results
        results['first_coordinate'].append(first_coordinate)
        results['second_coordinate'].append(second_coordinate)
        results['third_coordinate'].append(third_coordinate)
        results['gamma1'].append(popt[0])
        results['gamma2'].append(popt[1])
        results['k'].append(np.exp(popt[2]))
        results['r2'].append(r2)
        
# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(f"SEPresults_shots{shots}.csv", index=False)

plt.tight_layout()
plt.savefig(f"SEPfit_grid_shots{shots}.pdf".replace('|', '_').replace('/', '_').replace('\\', '_').replace('%', '_').replace('*', '_').replace('>', '_').replace(' ', ''), dpi=300)

# 4D Pedestal FIT

In [ ]:
# Load the dataframe
df = pd.read_csv(file_path)

# Label column name
label_column_name = 'label'

# Class
class_A_labels = [label for label in all_labels if (("developed" in label.lower()) and ('eda' in label.lower()) and ('elmy' not in label.lower()))]
print(class_A_labels)

class_B_labels = [label for label in all_labels if (("elmy" in label.lower()) and ('early' not in label.lower()))]
print(class_B_labels)



shots = [
    # 36257,
    36157,
    36124,
    37193,
    36349,
    36151,

    38049, 
    38066, 
    38067, # Shot todo comido?
    38742, 
    38743, 
    38744,

    36980,
    35970,
    38808,
]

# Combine class A and class B labels
selected_labels = class_A_labels + class_B_labels

# Filter dataframe based on selected labels
df = df[df['shot'].isin(shots)]
df = df[df[label_column_name].isin(selected_labels)]

# Add new columns to the dataframe
# df['1-Shapiro'] = 1 - df['|shapiro(Ipolsola>-rolling_0.999_quantile_50ms(-Ipolsola))']
df['1-Shapiro'] = 1 - df['MAC|shapiro(Ipolsola)']

# Apply a threshold to the new column
df = df[df['1-Shapiro'] < 0.03]


In [ ]:
# Select which columns in the dataframe will be the "features", i.e., the coordinates for the SVM classification (choose 2)

# Set the weighing method: shot or point
weighing = 'shot'
# weighing = 'point'

first_coordinate = 'UVS|D_tot'

second_coordinate = "FPC|IpiFP"

possible_third_coordinates = [
    "|P_TOT-Pr_sep",
    "TOT|P_TOT",
]

possible_fourth_coordinates = [
    '|ne0',
    '|ne95',
    '|pe0',
    '|pe95',
    '|Te0',
    '|Te95',
]

# possible_fourth_coordinates = [
#     'TOT|H-1_corr',
#     'TOT|Wmhd',
#     'TOT|tau_tot',
#     '|p_avg*tau_tot',
# ]

results = {
    'first_coordinate': [],
    'second_coordinate': [],
    'third_coordinate': [],
    'fourth_coordinate': [],
    'gamma1': [],
    'gamma2': [],
    'gamma3': [],
    'k': [],
    'r2': [],
}

fig, axes = plt.subplots(len(possible_fourth_coordinates), 2, figsize=(6, 12))  

axes = axes.T.flatten()
ii = -1

for third_coordinate in possible_third_coordinates:
    for fourth_coordinate in possible_fourth_coordinates:

        ii += 1

        # Select features and labels
        X = df[[first_coordinate, second_coordinate, third_coordinate, fourth_coordinate, "shot"]]

        X[first_coordinate] = np.log(X[first_coordinate])
        X[second_coordinate] = np.log(X[second_coordinate])
        X[third_coordinate] = np.log(X[third_coordinate])
        X[fourth_coordinate] = np.log(X[fourth_coordinate])

        # Check for nans
        mask = \
            X[first_coordinate].isnull()   | \
            X[second_coordinate].isnull()  | \
            X[third_coordinate].isnull()   | \
            X[fourth_coordinate].isnull()  | \
            np.isinf(X[first_coordinate])  | \
            np.isinf(X[second_coordinate]) | \
            np.isinf(X[third_coordinate])  | \
            np.isinf(X[fourth_coordinate])
        X = X[~mask]



        # 3D plot
        # fig = plt.figure(figsize=(6, 6))

        # ax = fig.add_subplot(111, projection='3d')
        # ax.scatter(
        #     X[first_coordinate],
        #     X[second_coordinate],
        #     X[third_coordinate],
        #     c=X[fourth_coordinate], 
        #     marker='o')
        # ax.set_xlabel("log" + first_coordinate)
        # ax.set_ylabel("log" + second_coordinate)
        # ax.set_zlabel("log" + third_coordinate)

        # plt.savefig(f"3D_{first_coordinate}_{second_coordinate}_{third_coordinate}_shots{shots}.pdf".replace('|', '_').replace('/', '_').replace('\\', '_').replace('%', '_').replace('*', '_').replace('>', '_'), dpi=300)




        sigma = np.ones(len(X[first_coordinate]))

        for i, shot in enumerate(shots):
            sigma[X['shot'] == shot] *= (len(sigma[X['shot'] == shot]))**0.5
            # print(f"{shot}: {len(sigma[X['shot'] == shot])}")

        # Define a power-law fitting function by fitting the log of the data
        def power_law(X, gamma1, gamma2, gamma3, logk):
            x1, x2, x3 = X
            return x1 * gamma1 + x2 * gamma2 + x3 * gamma3 + logk

        # Fit a power-law to the data
        from scipy.optimize import curve_fit

        # try:
        

        popt, pcov = curve_fit(
            power_law, 
            (
                X[first_coordinate], 
                X[second_coordinate], 
                X[third_coordinate]
            ), 
            X[fourth_coordinate],
            p0=[1, 1, 1, 1],
            sigma=sigma if weighing == 'shot' else None,
            )
        # except RuntimeError:
        #     # Optionally, use the analytical least squares solution
        #     # Resample the data to have the same weight per shot
        #     popt = analytical_least_squares(X[first_coordinate], X[second_coordinate], X[third_coordinate])
        #     pcov = np.zeros((3, 3))

        # Given the optimal parameters, calculate Pearson's R-squared
        xx = \
            np.exp(X[first_coordinate])**popt[0] * \
            np.exp(X[second_coordinate])**popt[1] * \
            np.exp(X[third_coordinate])**popt[2]
        yy = np.exp(X[fourth_coordinate])

        if weighing == 'shot':
            def _m(x, w):
                """Weighted Mean"""
                return np.sum(x * w) / np.sum(w)

            def _cov(x, y, w):
                """Weighted Covariance"""
                return np.sum(w * (x - _m(x, w)) * (y - _m(y, w))) / np.sum(w)

            def _corr(x, y, w):
                """Weighted Correlation"""
                return _cov(x, y, w) / np.sqrt(_cov(x, x, w) * _cov(y, y, w))
            
            r2 = _corr(xx, yy, sigma**-2)**2

        else:
            r2 = np.corrcoef(xx, yy)[0, 1] ** 2

        # Plot the data and the power-law fit
        # plt.figure(figsize=(3, 3))
        plt.sca(axes[ii])
        plt.plot(xx, yy, 'o', label='Data', alpha=0.1,)
        plt.plot(
            np.sort(xx), 
            np.sort(xx)*np.exp(popt[3]), 
            )
        plt.xlabel(
            f"{first_coordinate}^{popt[0]:.2f} *" + \
            f"{second_coordinate}^{popt[1]:.2f} *" + \
            f"{third_coordinate}^{popt[2]:.2f}",
            labelpad=10,
            )
        plt.ylabel(fourth_coordinate)

        plt.grid(False, which='minor')

        plt.title(
            # f"{shots}\n" + 
            f"k = {np.exp(popt[3]):.2e}\n" 
            # f"Same weight per {'POINT' if (sigma == 1.).all() else 'SHOT'}\n" +
            #f"Covariance matrix for the exponents:\n{pcov[0:3, 0:3]}\n" + 
            #f"Coefficient of determination r^2: {r2:.2f}"
        )
        # plt.tight_layout()
        # plt.savefig(f"2D_{first_coordinate}_{second_coordinate}_{third_coordinate}_shots{shots}.pdf".replace('|', '_').replace('/', '_').replace('\\', '_').replace('%', '_').replace('*', '_').replace('>', '_'), dpi=300)

        # Save the results
        results['first_coordinate'].append(first_coordinate)
        results['second_coordinate'].append(second_coordinate)
        results['third_coordinate'].append(third_coordinate)
        results['fourth_coordinate'].append(fourth_coordinate)
        results['gamma1'].append(popt[0])
        results['gamma2'].append(popt[1])
        results['gamma3'].append(popt[2])
        results['k'].append(np.exp(popt[3]))
        results['r2'].append(r2)
        
# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(f"results_3D_shots{shots}.csv", index=False)
results_df.to_csv(f"results_3D_shots_same_weight_per_{'shot' if weighing == 'shot' else 'point'}.csv", index=False)

plt.tight_layout()
plt.savefig(f"fit_grid_3D_shots{shots}.pdf".replace('|', '_').replace('/', '_').replace('\\', '_').replace('%', '_').replace('*', '_').replace('>', '_').replace(' ', ''), dpi=300)

# 4D Sep FIT

In [ ]:
# Select which columns in the dataframe will be the "features", i.e., the coordinates for the SVM classification (choose 2)

# Set the weighing method: shot or point
weighing = 'shot'
# weighing = 'point'

first_coordinate = 'UVS|D_tot'

second_coordinate = "FPC|IpiFP"

possible_third_coordinates = [
    "|P_TOT-Pr_sep",
    "TOT|P_TOT",
]
'''
possible_fourth_coordinates = [
    '|ne0',
    '|ne95',
    '|pe0',
    '|pe95',
    '|Te0',
    '|Te95',
]
'''

possible_fourth_coordinates = [
    '$T_{sep}$ (eV)',
    '$n_{sep}$ ($10^{19}$ m$^{-3}$)',
    '$p_{sep}$ (kPa)',
    '$T_{sep}^{(m)}$ (eV)',
    '$n_{sep}^{(m)}$ ($10^{19}$ m$^{-3}$)',
    '$p_{sep}^{(m)}$ (kPa)',
    '$p_{sep}$ (kPa)',    
]
# possible_fourth_coordinates = [
#     'TOT|H-1_corr',
#     'TOT|Wmhd',
#     'TOT|tau_tot',
#     '|p_avg*tau_tot',
# ]

results = {
    'first_coordinate': [],
    'second_coordinate': [],
    'third_coordinate': [],
    'fourth_coordinate': [],
    'gamma1': [],
    'gamma2': [],
    'gamma3': [],
    'k': [],
    'r2': [],
}

fig, axes = plt.subplots(len(possible_fourth_coordinates), 2, figsize=(6, 12))  

axes = axes.T.flatten()
ii = -1

for third_coordinate in possible_third_coordinates:
    for fourth_coordinate in possible_fourth_coordinates:

        ii += 1

        # Select features and labels
        X = df[[first_coordinate, second_coordinate, third_coordinate, fourth_coordinate, "shot"]]

        X[first_coordinate] = np.log(X[first_coordinate])
        X[second_coordinate] = np.log(X[second_coordinate])
        X[third_coordinate] = np.log(X[third_coordinate])
        X[fourth_coordinate] = np.log(X[fourth_coordinate])

        # Check for nans
        mask = \
            X[first_coordinate].isnull()   | \
            X[second_coordinate].isnull()  | \
            X[third_coordinate].isnull()   | \
            X[fourth_coordinate].isnull()  | \
            np.isinf(X[first_coordinate])  | \
            np.isinf(X[second_coordinate]) | \
            np.isinf(X[third_coordinate])  | \
            np.isinf(X[fourth_coordinate])
        X = X[~mask]



        # 3D plot
        # fig = plt.figure(figsize=(6, 6))

        # ax = fig.add_subplot(111, projection='3d')
        # ax.scatter(
        #     X[first_coordinate],
        #     X[second_coordinate],
        #     X[third_coordinate],
        #     c=X[fourth_coordinate], 
        #     marker='o')
        # ax.set_xlabel("log" + first_coordinate)
        # ax.set_ylabel("log" + second_coordinate)
        # ax.set_zlabel("log" + third_coordinate)

        # plt.savefig(f"3D_{first_coordinate}_{second_coordinate}_{third_coordinate}_shots{shots}.pdf".replace('|', '_').replace('/', '_').replace('\\', '_').replace('%', '_').replace('*', '_').replace('>', '_'), dpi=300)




        sigma = np.ones(len(X[first_coordinate]))

        for i, shot in enumerate(shots):
            sigma[X['shot'] == shot] *= (len(sigma[X['shot'] == shot]))**0.5
            # print(f"{shot}: {len(sigma[X['shot'] == shot])}")

        # Define a power-law fitting function by fitting the log of the data
        def power_law(X, gamma1, gamma2, gamma3, logk):
            x1, x2, x3 = X
            return x1 * gamma1 + x2 * gamma2 + x3 * gamma3 + logk

        # Fit a power-law to the data
        from scipy.optimize import curve_fit

        # try:
        

        popt, pcov = curve_fit(
            power_law, 
            (
                X[first_coordinate], 
                X[second_coordinate], 
                X[third_coordinate]
            ), 
            X[fourth_coordinate],
            p0=[1, 1, 1, 1],
            sigma=sigma if weighing == 'shot' else None,
            )
        # except RuntimeError:
        #     # Optionally, use the analytical least squares solution
        #     # Resample the data to have the same weight per shot
        #     popt = analytical_least_squares(X[first_coordinate], X[second_coordinate], X[third_coordinate])
        #     pcov = np.zeros((3, 3))

        # Given the optimal parameters, calculate Pearson's R-squared
        xx = \
            np.exp(X[first_coordinate])**popt[0] * \
            np.exp(X[second_coordinate])**popt[1] * \
            np.exp(X[third_coordinate])**popt[2]
        yy = np.exp(X[fourth_coordinate])

        if weighing == 'shot':
            def _m(x, w):
                """Weighted Mean"""
                return np.sum(x * w) / np.sum(w)

            def _cov(x, y, w):
                """Weighted Covariance"""
                return np.sum(w * (x - _m(x, w)) * (y - _m(y, w))) / np.sum(w)

            def _corr(x, y, w):
                """Weighted Correlation"""
                return _cov(x, y, w) / np.sqrt(_cov(x, x, w) * _cov(y, y, w))
            
            r2 = _corr(xx, yy, sigma**-2)**2

        else:
            r2 = np.corrcoef(xx, yy)[0, 1] ** 2

        # Plot the data and the power-law fit
        # plt.figure(figsize=(3, 3))
        plt.sca(axes[ii])
        plt.plot(xx, yy, 'o', label='Data', alpha=0.1,)
        plt.plot(
            np.sort(xx), 
            np.sort(xx)*np.exp(popt[3]), 
            )
        plt.xlabel(
            f"{first_coordinate}^{popt[0]:.2f} *" + \
            f"{second_coordinate}^{popt[1]:.2f} *" + \
            f"{third_coordinate}^{popt[2]:.2f}",
            labelpad=10,
            )
        plt.ylabel(fourth_coordinate)

        plt.grid(False, which='minor')

        plt.title(
            # f"{shots}\n" + 
            f"k = {np.exp(popt[3]):.2e}\n" 
            # f"Same weight per {'POINT' if (sigma == 1.).all() else 'SHOT'}\n" +
            #f"Covariance matrix for the exponents:\n{pcov[0:3, 0:3]}\n" + 
            #f"Coefficient of determination r^2: {r2:.2f}"
        )
        # plt.tight_layout()
        # plt.savefig(f"2D_{first_coordinate}_{second_coordinate}_{third_coordinate}_shots{shots}.pdf".replace('|', '_').replace('/', '_').replace('\\', '_').replace('%', '_').replace('*', '_').replace('>', '_'), dpi=300)

        # Save the results
        results['first_coordinate'].append(first_coordinate)
        results['second_coordinate'].append(second_coordinate)
        results['third_coordinate'].append(third_coordinate)
        results['fourth_coordinate'].append(fourth_coordinate)
        results['gamma1'].append(popt[0])
        results['gamma2'].append(popt[1])
        results['gamma3'].append(popt[2])
        results['k'].append(np.exp(popt[3]))
        results['r2'].append(r2)
        
# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(f"results_3D_shots{shots}.csv", index=False)
results_df.to_csv(f"results_3D_shots_same_weight_per_{'shot' if weighing == 'shot' else 'point'}.csv", index=False)

plt.tight_layout()
plt.savefig(f"fit_grid_3D_shots{shots}.pdf".replace('|', '_').replace('/', '_').replace('\\', '_').replace('%', '_').replace('*', '_').replace('>', '_').replace(' ', ''), dpi=300)

# Jackknife for 4D pedestal fit

In [ ]:
# Select which columns in the dataframe will be the "features", i.e., the coordinates for the SVM classification (choose 2)

first_coordinate = 'UVS|D_tot'

second_coordinate = "FPC|IpiFP"

possible_third_coordinates = [
    "|P_TOT-Pr_sep",
    "TOT|P_TOT",
]

# possible_fourth_coordinates = [
#     '|ne0',
#     '|ne95',
#     '|pe0',
#     '|pe95',
#     '|Te0',
#     '|Te95',
# ]

possible_fourth_coordinates = [
    '|ne0',
    '|ne10',
    '|ne20',
    '|ne30',
    '|ne40',
    '|ne50',
    '|ne60',
    '|ne70',
    '|ne80',
    '|ne90',
    '|ne95',
    '|ne100',
    '|pe0',
    '|pe10',
    '|pe20',
    '|pe30',
    '|pe40',
    '|pe50',
    '|pe60',
    '|pe70',
    '|pe80',
    '|pe90',
    '|pe95',
    '|pe100',
    '|Te0',
    '|Te10',
    '|Te20',
    '|Te30',
    '|Te40',
    '|Te50',
    '|Te60',
    '|Te70',
    '|Te80',
    '|Te90',
    '|Te95',
    '|Te100',
    'TOT|H-1_corr',
    'TOT|Wmhd',
    'TOT|tau_tot',
    '|p_avg*tau_tot',
]

results = {
    'first_coordinate': [],
    'second_coordinate': [],
    'third_coordinate': [],
    'fourth_coordinate': [],
    'gamma1': [],
    'gamma2': [],
    'gamma3': [],
    'k': [],
    'r2': [],
    'removed_shot': [],
    'logk_raised': [],
}

# fig, axes = plt.subplots(len(possible_fourth_coordinates), 2, figsize=(6, 12))  

# axes = axes.T.flatten()
# ii = -1

for removed_shot in shots + [99999]:

    for third_coordinate in possible_third_coordinates:
        for fourth_coordinate in possible_fourth_coordinates:

            # ii += 1

            # Select features and labels
            X = df[[first_coordinate, second_coordinate, third_coordinate, fourth_coordinate, "shot"]]

            # Shot mask
            mask = X['shot'] != removed_shot
            X = X[mask]

            X[first_coordinate] = np.log(X[first_coordinate])
            X[second_coordinate] = np.log(X[second_coordinate])
            X[third_coordinate] = np.log(X[third_coordinate])
            X[fourth_coordinate] = np.log(X[fourth_coordinate])

            # Check for nans
            mask = \
                X[first_coordinate].isnull()   | \
                X[second_coordinate].isnull()  | \
                X[third_coordinate].isnull()   | \
                X[fourth_coordinate].isnull()  | \
                np.isinf(X[first_coordinate])  | \
                np.isinf(X[second_coordinate]) | \
                np.isinf(X[third_coordinate])  | \
                np.isinf(X[fourth_coordinate])
            X = X[~mask]



            # 3D plot
            # fig = plt.figure(figsize=(6, 6))

            # ax = fig.add_subplot(111, projection='3d')
            # ax.scatter(
            #     X[first_coordinate],
            #     X[second_coordinate],
            #     X[third_coordinate],
            #     c=X[fourth_coordinate], 
            #     marker='o')
            # ax.set_xlabel("log" + first_coordinate)
            # ax.set_ylabel("log" + second_coordinate)
            # ax.set_zlabel("log" + third_coordinate)

            # plt.savefig(f"3D_{first_coordinate}_{second_coordinate}_{third_coordinate}_shots{shots}.pdf".replace('|', '_').replace('/', '_').replace('\\', '_').replace('%', '_').replace('*', '_').replace('>', '_'), dpi=300)




            sigma = np.ones(len(X[first_coordinate]))

            for i, shot in enumerate(shots):
                sigma[X['shot'] == shot] *= (len(sigma[X['shot'] == shot]))**0.5
                # print(f"{shot}: {len(sigma[X['shot'] == shot])}")

            # Define a power-law fitting function by fitting the log of the data
            def power_law(X, gamma1, gamma2, gamma3, logk_raised):
                x1, x2, x3 = X
                return x1 * gamma1 + x2 * gamma2 + x3 * gamma3 + logk_raised

            # Fit a power-law to the data
            from scipy.optimize import curve_fit

            # try:
            popt, pcov = curve_fit(
                power_law, 
                (
                    X[first_coordinate], 
                    X[second_coordinate], 
                    X[third_coordinate]
                ), 
                X[fourth_coordinate],
                p0=[1, 1, 1, 1],
                sigma=sigma if weighing == 'shot' else None,
                )

            # Given the optimal parameters, calculate Pearson's R-squared
            xx = \
                np.exp(X[first_coordinate])**popt[0] * \
                np.exp(X[second_coordinate])**popt[1] * \
                np.exp(X[third_coordinate])**popt[2]
            yy = np.exp(X[fourth_coordinate])

            if weighing == 'shot':
                def _m(x, w):
                    """Weighted Mean"""
                    return np.sum(x * w) / np.sum(w)

                def _cov(x, y, w):
                    """Weighted Covariance"""
                    return np.sum(w * (x - _m(x, w)) * (y - _m(y, w))) / np.sum(w)

                def _corr(x, y, w):
                    """Weighted Correlation"""
                    return _cov(x, y, w) / np.sqrt(_cov(x, x, w) * _cov(y, y, w))
                
                r2 = _corr(xx, yy, sigma**-2)**2

            else:
                r2 = np.corrcoef(xx, yy)[0, 1] ** 2


            # Save the results
            results['first_coordinate'].append(first_coordinate)
            results['second_coordinate'].append(second_coordinate)
            results['third_coordinate'].append(third_coordinate)
            results['fourth_coordinate'].append(fourth_coordinate)
            results['gamma1'].append(popt[0])
            results['gamma2'].append(popt[1])
            results['gamma3'].append(popt[2])
            results['logk_raised'].append(popt[3])
            results['k'].append(np.exp(popt[3] / (popt[0] + popt[1] + popt[2])))
            results['r2'].append(r2)
            results['removed_shot'].append(removed_shot)
            
# Save results to a CSV file
results_df = pd.DataFrame(results)
results_df.to_csv(f"results_3D_shots{shots}.csv", index=False)

with open(f"jackknife_pedestal_results_same_weight_per_{weighing}.csv", 'w') as f:

    f.write("Third Coordinate, Fourth Coordinate, Parameter, Jackknife Estimate, Standard Deviation, Standard Error, Half Confidence Interval\n")

    for third_coordinate in possible_third_coordinates:
        for fourth_coordinate in possible_fourth_coordinates:

            # Calculate jackknife estimates for each parameter
            for parameter in ["gamma1", "gamma2", "gamma3", "logk_raised", "k", "r2"]:

                results = results_df[
                    (results_df['third_coordinate'] == third_coordinate) &
                    (results_df['fourth_coordinate'] == fourth_coordinate)
                ].sort_values(by='removed_shot', ignore_index=True, ascending=True)

                # Use results to calculate jackknife estimates
                w0_tilde_i = len(shots) * np.array(results[parameter])[: -1] - (len(shots) - 1) * np.array(results[parameter])[: -1]
                w0_tilde = np.mean(w0_tilde_i)

                variance_tilde = 1 / (len(shots)-1) * np.sum((w0_tilde_i - w0_tilde)**2)
                std_tilde = (variance_tilde)**0.5
                half_width_95 = 1.96 * np.sqrt(1/len(shots) * variance_tilde)
                confidence_interval = [w0_tilde - 1.96 * np.sqrt(1/len(shots) * variance_tilde), w0_tilde + 1.96 * np.sqrt(1/len(shots) * variance_tilde)]

                f.write(f"{third_coordinate}, {fourth_coordinate}, {parameter}, {w0_tilde}, {std_tilde}, {std_tilde / (len(shots))**0.5}, {half_width_95}\n".replace('|', '').replace('_', ''))	
